<a href="https://colab.research.google.com/github/krishna25045678/Cleaning_home/blob/main/ProblemSet4_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning and Neural Networks

## Tutorial 4: Semi-supervised learning

Welcome to the fourth tutorial for Machine Learning and Neural Netowkrs. In this session, we will explore how to apply Semi-supervised learning, through increasingly complex problems.


Throughout this session, we will re-use the same dataset. This is the California Housing dataset, which you can download as follows:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import fetch_openml

data = fetch_openml(name='adult', version=2)
x, y = data.data, data.target#

x = pd.get_dummies(x)
y = LabelEncoder().fit_transform(y)

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train)

print(x_train.shape)

is this a classification or regression task?

Since semi-supervised learning uses only a small number of labelled data, we will create a duplicate training dataset where only a few examples are labelled. However, we will keep the full dataset, as at the end we will compare the performance with supervised learning.

In [ ]:
# Number of labeled samples
n_labeled = 1000

# Create a copy of the training data
x_train_copy = x_train.copy().to_numpy()
y_train_copy = y_train.copy()

# Randomly select indices for labeled data
indices = np.random.permutation(len(x_train_copy))
x_labeled = x_train_copy[indices[:n_labeled]]
y_labeled = y_train_copy[indices[:n_labeled]]

# The rest will be unlabeled
x_unlabeled = x_train_copy[indices[n_labeled:]]
y_unlabeled = y_train_copy[indices[n_labeled:]]  # This will not be used in training

You can use this dataset throughout the semi-supervised tasks.

#### Problem 1: Self-training

Implement self-training using your model of choice from scikit-learn. You will need to define the self-training loop, initialise a model, train it, and evaluate its accuracy.

In [ ]:
# Enter you solution here
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Optional: Standardize the data
scaler = StandardScaler()
x_labeled = scaler.fit_transform(x_labeled)
x_unlabeled = scaler.transform(x_unlabeled)
x_test = scaler.transform(x_test)

# Implement self-training
def self_training(x_labeled, y_labeled, x_unlabeled, base_model, threshold=0.9, max_iter=10):
    model = base_model
    for i in range(max_iter):
        # Train the model on the labeled data
        model.fit(x_labeled, y_labeled)

        # Predict labels for the unlabeled data
        y_unlabeled_pred = model.predict(x_unlabeled)
        y_unlabeled_prob = model.decision_function(x_unlabeled)

        # Select the most confident predictions
        print(y_unlabeled_prob.shape)
        confident_indices = np.where(y_unlabeled_prob > threshold)[0]
        if len(confident_indices) == 0:
            break

        # Add the confident predictions to the labeled set
        x_labeled = np.vstack((x_labeled, x_unlabeled[confident_indices]))
        y_labeled = np.hstack((y_labeled, y_unlabeled_pred[confident_indices]))

        # Remove the confident predictions from the unlabeled set
        x_unlabeled = np.delete(x_unlabeled, confident_indices, axis=0)

    return model

# Initialize the base model - we are using SVM here but students can use any other model
base_model = SVC(probability=True)

# Perform self-training
model = self_training(x_labeled, y_labeled, x_unlabeled, base_model)

# Evaluate the model
y_test_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {accuracy:.4f}")

#### Problem 2: Label propagation

Using the same dataset, try to apply Label Propagation. Why does it crash? What do you need to do to get around this?

In [ ]:
# Enter you solution here
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import accuracy_score

# Make a portion of the training labels unlabeled
n_labeled = int(0.1 * len(y_train))  # Use 10% of the training data as labeled
y_train_unlabeled = np.copy(y_train)
y_train_unlabeled[n_labeled:] = -1  # Use -1 to indicate unlabeled data - note how this is different from the previous examples

# Step 3: Implement label propagation
label_prop_model = LabelPropagation(kernel='knn') # Use a k-NN kernel, otherwise it is likely to crash - the graph is too big if it is fully connected!
label_prop_model.fit(x_train, y_train_unlabeled)

# Step 4: Evaluate the model
y_test_pred = label_prop_model.predict(x_test)
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {accuracy:.4f}")

#### Problem 3: Model optimisation

For either the self-training or label propagation cases, try to perform hyperparameter optimisation using GridSearchCV. This can take a little while.

In [ ]:
# Enter you solution here
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
# We will search for the best value of n_neighbors, since we found we had to use the knn kernel above or it would crash
# We can't test too many combinations since we don't have much time in the tutorial
param_grid = {
    "n_neighbors": [3, 7, 11, 15],
}

# Make a portion of the training labels unlabeled
n_labeled = int(0.1 * len(y_train))  # Use 10% of the training data as labeled
y_train_unlabeled = np.copy(y_train)
y_train_unlabeled[n_labeled:] = -1  # Use -1 to indicate unlabeled data - note how this is different from the previous examples


# Initialize the Label Propagation model
label_prop_model = LabelPropagation(kernel='knn')

# Initialize GridSearchCV
grid_search = GridSearchCV(label_prop_model, param_grid, cv=5, scoring='accuracy')

# Fit the model
grid_search.fit(x_train, y_train_unlabeled)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(x_test)
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {accuracy:.4f}")

#### Supervised comparison

Finally, train a supervised classifier on the full dataset and see how it compares. Since the dataset is quite large, this can be a very slow step depending on your choice of classifier.

In [ ]:
# Enter you solution here
# We use the same classifier as in the self-training example for a fair comparison, but use all the data
# You can try different classifiers and hyperparameters
base_model = SVC(probability=True)

# Train the model on all the data
base_model.fit(x_train, y_train)

# Evaluate the model
y_test_pred = base_model.predict(x_test)
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {accuracy:.4f}")

#### Extension problems

For your own understanding, it would be a good idea to continue to play with these models in your own time. You could test what happens when you change your choice of underlying model (e.g. use a random forest if you originally tried SVM) or test on a different dataset. You could also try to implement co-training or a generative model (using the EM algorithm).